In [ ]:
#!pip install openpyxl
#!pip install pandas

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

In [18]:
# Mapping OpenPyXL cell data_type codes to standard data type names
OPENPYXL_TYPE_MAP = {
    'n': 'number',   # Numeric (float, int)
    's': 'string',   # String
    'b': 'boolean',  # Boolean
    'f': 'formula',  # Formula (can also have a value if data_only=True)
    'e': 'error',    # Error
    'd': 'date',     # Date (not always present, use cell.is_date)
    'null': 'empty'  # Empty cell (not standard, sometimes None is used)
}

def openpyxl_code_to_type(code):
    """
    Convert an OpenPyXL cell data_type code to a standard data type description.
    If code is unknown, returns 'unknown'.
    """
    return OPENPYXL_TYPE_MAP.get(code, 'unknown')

In [19]:
import openpyxl

def read_all_sheets(file_path):
    workbook = openpyxl.load_workbook(file_path, data_only=True)

    all_data = {}

    for sheet_name in workbook.sheetnames:
        sheet = workbook[sheet_name]
        sheet_data = []

        for row in sheet.iter_rows(values_only=False):  # Get real cell objects now!
            row_data = []
            for cell in row:
                row_data.append({
                    "value": cell.value,
                    "py_type": type(cell.value).__name__,
                    #"xl_type": cell.data_type  # Openpyxl's code: 'n', 's', 'b', etc.
                    "xl_type": openpyxl_code_to_type(cell.data_type)
                })
            sheet_data.append(row_data)

        all_data[sheet_name] = sheet_data

    return all_data

def print_all_data(all_data):
    for sheet_name, rows in all_data.items():
        print(f"\n=== Sheet: {sheet_name} ===")
        for row in rows:
            for cell in row:
                print(cell)

def create_db(all_data):
    cell_db = []
    for sheet_name, rows in all_data.items():
        for row_idx, row in enumerate(rows, start=1):
            for col_idx, cell_record in enumerate(row, start=1):
                append_this = {
                    "sheet": sheet_name,
                    "row": row_idx,
                    "column": col_idx,
                    "value": cell_record["value"],
                    "py_type": cell_record["py_type"],
                    "xl_type": cell_record["xl_type"]
                }
                cell_db.append(append_this)
    return cell_db


In [20]:
#file_path = "/content/drive/MyDrive/Colab Notebooks/CellDB/data.xlsx"
file_path = './Data.xlsx'
all_data = read_all_sheets(file_path)
cdb = create_db(all_data)

In [21]:
cdb[0]

{'sheet': 'AgeGenderSheet',
 'row': 1,
 'column': 1,
 'value': None,
 'py_type': 'NoneType',
 'xl_type': 'number'}

In [29]:
import pandas as pd
cd_df = pd.DataFrame(cdb)

In [23]:
ags_df = cd_df[cd_df['sheet'] == 'AgeGenderSheet']

In [31]:
unique_combinations = cd_df[['py_type', 'xl_type']].drop_duplicates().reset_index(drop=True)


print(unique_combinations)

    py_type xl_type
0  NoneType  number
1       str  string
2       int  number
3  datetime    date
